## Initialisation du modèle

Ici, on charge tous les packages, les variables, et on choisis le modele à utiliser

In [2]:
import os
import datetime
import random
import numpy as np
import pandas as pd
import torch
import esm
import joblib
from sklearn.metrics import mean_absolute_error, precision_score, recall_score, accuracy_score, confusion_matrix

# 1. Configuration matérielle (Apple Silicon)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Hardware détecté : {device}")

# 2. Chargement du modèle de langage ESM-2
print("Chargement d'ESM-2 en cours...")
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()
model.to(device)

# 3. Chargement interactif du modèle SVR
dossier_sauvegarde = "modeles_sauvegardes"

# Affichage des modèles disponibles pour faciliter le copier-coller
if os.path.exists(dossier_sauvegarde):
    fichiers_dispos = [f for f in os.listdir(dossier_sauvegarde) if f.endswith('.joblib')]
    if fichiers_dispos:
        print(f"\nModèles disponibles dans '{dossier_sauvegarde}' :")
        for f in fichiers_dispos:
            print(f" 📂 {f}")
    else:
        print(f"\nAucun modèle .joblib trouvé dans le dossier '{dossier_sauvegarde}'.")
else:
    print(f"\nLe dossier '{dossier_sauvegarde}' n'existe pas encore.")

nom_modele_charge = input("\nEntrez le nom exact du modèle SVR à charger (ex: modele_svr_...).joblib : ")
chemin_modele = os.path.join(dossier_sauvegarde, nom_modele_charge)

regr = joblib.load(chemin_modele)
print(f" Modèle SVR chargé avec succès : {nom_modele_charge}")


# 4. Séquences de référence et variables globales
VP1_WT_1 = "MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLVLPGYKYLGPFNGLDKGEPVNEADAAALEHDKAYDRQLDSGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRVLEPLGLVEEPVKTAPGKKRPVEHSPVEPDSSSGTGKAGQQPARKRLNFGQTGDADSVPDPQPLGQPPAAPSGLGTNTMATGSGAPMADNNEGADGVGNSSGNWHCDSTWMGDRVITTSTRTWALPTYNNHLYKQISSQSGASNDNHYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTQNDGTTTIANNLTSTVQVFTDSEYQLPYVLGSAHQGCLPPFPADVFMVPQYGYLTLNNGSQAVGRSSFYCLEYFPSQMLRTGNNFTFSYTFEDVPFHSSYAHSQSLDRLMNPLIDQYLYYLSRTNTPSGTTTQSRLQFSQAGASDIRDQSRNWLPGPCYRQQRVSKTSADNNNSEYSWTGATKYHLNGRDSLVNPGPAMASHKDDEEKFFPQSGVLIFGKQGSEKTNVDIEKVMIT"
WT_boucle28 = "DEEEIRTTNPVATEQYGSVSTNLQRGNR"
VP1_WT_2 = "QAATADVNTQGVLPGMVWQDRDVYLQGPIWAKIPHTDGHFHPSPLMGGFGLKHPPPQILIKNTPVPANPSTTFSAAKFASFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYNKSVNVDFTVDTNGVYSEPRPIGTRYLTRNL"
VP1_WT = VP1_WT_1 + WT_boucle28 + VP1_WT_2
ACIDES_AMINES = "ACDEFGHIKLMNPQRSTVWY"

print(" Initialisation terminée. L'environnement de travail est prêt.")




Hardware détecté : mps
Chargement d'ESM-2 en cours...

Modèles disponibles dans 'modeles_sauvegardes' :
 📂 modele_svr_50k.joblib
 📂 modele_svr_20260930_1113_ESM8M_100seq.joblib
 Modèle SVR chargé avec succès : modele_svr_50k.joblib
 Initialisation terminée. L'environnement de travail est prêt.


Ici, on teste sur un sous échantillon

In [3]:
import pandas as pd
import numpy as np

# 1. Charger les données (complet vs entraînement)
data_complet = pd.read_csv('Variants(complet).csv')
data_entrainement = pd.read_csv('Variants_echantillon.csv')

# 2. Filtrer pour exclure les séquences déjà utilisées
sequences_entrainement = data_entrainement['sequence'].tolist()
data_inedit = data_complet[~data_complet['sequence'].isin(sequences_entrainement)]

# 3. Choix du nombre de séquences à tester
nb_test = int(input("Nombre de séquences inédites à tester : "))
data_test = data_inedit.sample(n=nb_test)

# 4. Préparer les séquences entières et les scores réels
seq_test_brutes = data_test['sequence'].tolist()
scores_reels = data_test['viral_selection'].replace(-np.inf, -15).tolist()

liste_seq_test = []
for i in seq_test_brutes:
    boucle = i.upper().replace('*', '')
    liste_seq_test.append(VP1_WT_1 + boucle + VP1_WT_2)

# Préparation du tuple pour ESM-2
data_tuple_test = [(f"test_{i}", seq) for i, seq in enumerate(liste_seq_test)]
print(f"\n{nb_test} séquences inédites préparées avec succès.")


import torch
import numpy as np

# On prépare la liste pour stocker tous les vecteurs finaux
all_embeddings_test = []
batch_size = 16

print("Extraction des vecteurs ESM-2 par lots en cours...")

# On désactive le calcul du gradient pour économiser la mémoire globale
with torch.no_grad():

    # La boucle avance par sauts de 16
    for i in range(0, len(data_tuple_test), batch_size):

        # 1. Sélection et tokenisation du petit lot
        batch = data_tuple_test[i : i + batch_size]
        _, _, batch_tokens_test = batch_converter(batch)
        batch_tokens_test = batch_tokens_test.to(device)

        # 2. Passage dans ESM-2 (uniquement pour ces 16 séquences)
        results_test = model(batch_tokens_test, repr_layers=[6], return_contacts=False)
        embeddings_bruts_test = results_test["representations"][6]

        # 3. Mean Pooling pour ce lot
        for j, (_, seq) in enumerate(batch):
            longueur_reelle = len(seq)
            tranche_proteine = embeddings_bruts_test[j, 1 : longueur_reelle + 1]
            all_embeddings_test.append(tranche_proteine.mean(dim=0).cpu().numpy())

        # 4. Vidage de la mémoire du Mac avant le prochain lot
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

# Transformation finale en tableau Numpy
X_test = np.array(all_embeddings_test)

# 5. Prédiction avec le modèle SVR
print("Calcul des prédictions via le modèle SVR...")
scores_predits = regr.predict(X_test)
print("Prédictions terminées !")


import pandas as pd

# 1. Création du tableau de comparaison
comparaison_df = pd.DataFrame({
    'Séquence (boucle)': seq_test_brutes,
    'Score Réel': scores_reels,
    'Score Prédit': scores_predits
})

# 2. Calcul de l'erreur absolue pour évaluer la précision
comparaison_df['Différence absolue'] = abs(comparaison_df['Score Réel'] - comparaison_df['Score Prédit'])

# 3. Affichage du résultat, trié de la prédiction la plus précise à la moins précise
print("\n--- RÉSULTATS DU TEST D'INFÉRENCE ---")
print(comparaison_df.sort_values(by='Différence absolue').to_string(index=False))




import os
import datetime
import numpy as np
from sklearn.metrics import mean_absolute_error, precision_score, recall_score, accuracy_score, confusion_matrix

# 0. Connexion avec tes vraies données
y_real = comparaison_df['Score Réel'].values
y_pred = comparaison_df['Score Prédit'].values

# --- PARAMÈTRES DYNAMIQUES ---
# On extrait dynamiquement les infos du modèle chargé si la variable existe, sinon on indique "Inconnu"
fich_modele = globals().get('nom_modele_charge', 'Modèle personnalisé')

# Détection automatique de la taille d'entraînement si elle est inscrite dans le nom du fichier
if "100seq" in fich_modele:
    taille_entrainement = "100"
elif "50k" in fich_modele or "50000" in fich_modele:
    taille_entrainement = "50 000"
else:
    taille_entrainement = "Personnalisée / Inconnue"

modele_esm = "ESM-2 (8M paramètres - t6_8M_UR50D)"
modele_ml = "SVR (Scikit-Learn, C=1.0, epsilon=0.2)"
nb_test = len(y_real)
date_test = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

# 1. Statistiques continues (Régression)
mae = mean_absolute_error(y_real, y_pred)

# 2. Conversion en Booléen (Classification)
seuil = 0
y_real_bool = y_real > seuil
y_pred_bool = y_pred > seuil

# 3. Calcul des métriques biologiques
precision = precision_score(y_real_bool, y_pred_bool, zero_division=0)
rappel = recall_score(y_real_bool, y_pred_bool, zero_division=0)
accuracy = accuracy_score(y_real_bool, y_pred_bool)
tn, fp, fn, tp = confusion_matrix(y_real_bool, y_pred_bool).ravel()

# --- CRÉATION DU DOSSIER ET EXPORT ---
dossier = "rapports"
os.makedirs(dossier, exist_ok=True)

# Construction du texte du rapport avec traçabilité complète
texte_rapport = f"""--- CONDITIONS DU TEST ---
Date de l'analyse : {date_test}
Fichier modèle utilisé : {fich_modele}
Modèle d'extraction : {modele_esm}
Modèle d'apprentissage : {modele_ml}
Taille du jeu d'entraînement : {taille_entrainement} séquences
Taille de l'échantillon de test : {nb_test} séquences inédites

--- RAPPORT DE VIABILITÉ BIOLOGIQUE (Seuil > {seuil}) ---
Écart Absolu Moyen (MAE) : {mae:.3f}

Précision (Hit Rate) : {precision * 100:.1f}% -> Si le modèle dit OUI, il a raison dans {precision * 100:.1f}% des cas.
Rappel (Sensibilité) : {rappel * 100:.1f}% -> Le modèle détecte {rappel * 100:.1f}% de toutes les capsides viables.
Exactitude Globale   : {accuracy * 100:.1f}%

--- MATRICE DE CONFUSION ---
Vrais Positifs (Succès labo)       : {tp}
Faux Positifs (Échecs labo/Coût)   : {fp}
Vrais Négatifs (Déchets évités)    : {tn}
Faux Négatifs (Opportunités ratées): {fn}

--- DONNÉES BRUTES ---
"""

# Ajout du tableau trié à la suite du texte
df_trie = comparaison_df.sort_values(by='Différence absolue')
texte_rapport += df_trie.to_string(index=False)

# Affichage direct dans le notebook
print(texte_rapport)

# Sauvegarde avec un nom de fichier dynamique incluant la taille du test
nom_fichier_txt = f"rapport_inference_{nb_test}seq.txt"
chemin_txt = os.path.join(dossier, nom_fichier_txt)
with open(chemin_txt, "w", encoding="utf-8") as f:
    f.write(texte_rapport)

nom_fichier_csv = f"donnees_inference_{nb_test}seq.csv"
chemin_csv = os.path.join(dossier, nom_fichier_csv)
df_trie.to_csv(chemin_csv, index=False)

print(f"\n Rapport sauvegardé avec succès sous le nom '{nom_fichier_txt}' dans le dossier '{dossier}'.")


10 séquences inédites préparées avec succès.
Extraction des vecteurs ESM-2 par lots en cours...
Calcul des prédictions via le modèle SVR...
Prédictions terminées !

--- RÉSULTATS DU TEST D'INFÉRENCE ---
                     Séquence (boucle)  Score Réel  Score Prédit  Différence absolue
         AEEEIRTTNPVATEQYGSVSTNLNEGAeN    1.175699      1.281990            0.106291
   FEEEIRTTNPVATEQYGSCSpSQpALeViIePqDm   -5.437757     -5.202569            0.235188
    DEEEIRTTNPVAYEQYGqCVcSTEHQeYfGsNQs    0.419295      0.861395            0.442100
         YEEEIRTTNVVATEQYGSdVSTNLQRGHT   -5.392669     -4.931362            0.461308
DEqEaEISTTNPVAYECWGSVSWgDgQqQqDrRgDkEq   -3.737317     -4.869352            1.132035
    DEEEIRTTNPVATEQYGStIScTgNLsQRGgNRd    0.620605     -0.976274            1.596879
        DEEEIRTTNPTATEQFGEVSfTNLQRGdNR   -3.325482     -1.667674            1.657808
   DEEEICTTNPVATEQYGEVSTgGpLQdHqQgNmGh    0.500693     -1.414872            1.915565
          DEEEIRTTNPVATEQYGSVST